# 12: What does a compiler actually change?

**Level:** Intermediate  
**Before you start:** Notebook 01.  
**Resources:** CPU unless an optional remote step is enabled.

Remove redundant work while preserving the quantum result, then explore an optional hardware compiler.

Run each cell in order. All core calculations are written in this notebook.

## 1. Write an intentionally redundant circuit

Two adjacent X gates cancel. Two adjacent H gates also cancel. We can predict an optimization before inspecting the compiler output.

In [ ]:
import flagquantum as fq
import torch

q = fq.Circuit(2)
q.x(0)
q.x(0)
q.h(1)
q.h(1)
q.ry(0, 0.3)
q.cx(0, 1)
from flagquantum.core.ir import ensure_circuit_ir

before = ensure_circuit_ir(q)
after = fq.compile(q)
print("Before:", [(i.name, i.wires) for i in before.instructions])
print("After:", [(i.name, i.wires) for i in after.instructions])
assert len(after.instructions) < len(before.instructions)


## 2. Check meaning, not just gate count

A shorter circuit is useful only if it computes the intended result.

In [ ]:
original_state = fq.run(before).to_statevector()
compiled_state = fq.run(after).to_statevector()
torch.testing.assert_close(original_state, compiled_state, atol=1e-6, rtol=1e-6)
print("Numerical check passed")


## 3. Optional: compile for Quafu with QSteed

This step needs the installed plugin and access to current chip calibration. It can contact the platform, but it does not submit a hardware execution task. The final platform-returned circuit is only available after execution.

In [ ]:
compile_for_hardware = False
if compile_for_hardware:
    target = "quafu:Dongling"  # Confirm with your instructor.
    mapped = fq.compile(q, compiler="qsteed", target=target)
    print(mapped)
else:
    print("Hardware compilation skipped; local compiler exercise completed.")


## Make it yours

Construct redundant rotations and see what changes. Repeat with a trainable torch parameter: why must an optimizer preserve its gradient dependencies? Continue to notebook 17 to implement a small compiler extension yourself.